# Model Evaluation on FLEURS Benchmark

Evaluate fine-tuned Whisper models on the multilingual FLEURS dataset.

**Metrics:**
- WER (Word Error Rate)
- CER (Character Error Rate)

In [ ]:
import subprocess
import sys

# Install dependencies
packages = ["datasets>=2.13.0","transformers>=4.30.0","evaluate>=0.4.0","jiwer>=3.0.0"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from datasets import load_dataset
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

print("✓ Dependencies installed")


In [ ]:
# Load fine-tuned model
LANGUAGE = "mr"  # Change to "gu" or "hi"
MODEL_PATH = f"/content/drive/MyDrive/speech_agent_workspace/models/whisper-{LANGUAGE}-finetuned"

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)

print("✓ Found model at", MODEL_PATH)

# Load FLEURS split
fleurs = load_dataset("microsoft/fleurs", language=LANGUAGE, split="test")
print(f"Loaded FLEURS test split with {len(fleurs)} examples")

# Evaluate subset for speed
fleurs_subset = fleurs.select(range(min(100, len(fleurs))))

predictions = []
references = []

for i, sample in enumerate(fleurs_subset):
    if i % 10 == 0:
        print(f"Processing {i}/{len(fleurs_subset)}")
    audio = sample["audio"]
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features.to(device)
    predicted_ids = model.generate(input_features)
    pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    references.append(sample["sentence"])
    predictions.append(pred_text)

print("Calculating metrics...")
wer_score = wer(references, predictions)
cer_score = cer(references, predictions)
print(f"FLEURS Evaluation: WER={wer_score:.4f}, CER={cer_score:.4f}")


In [ ]:
import subprocess
import sys

packages = ["transformers>=4.30.0", "datasets>=2.13.0", "jiwer>=3.0.0", "evaluate>=0.4.0"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from transformers import pipeline, WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
from jiwer import wer, cer
import json
import torch

print("✓ Setup complete")